# Chapter 2: Solving the Inference Bottleneck with the Key-Value Cache

## Introduction

The inference bottleneck is one of the most critical challenges in deploying large language models like DeepSeek. This chapter explores how the key-value cache, a fundamental optimization technique addresses this bottleneck and serves as the foundation for more advanced attention mechanisms.

In autoregressive generation, each new token requires attention computations across all previous tokens in the sequence. Without optimizations, this would lead to:

1. Quadratically increasing computation time as sequence length grows
2. Redundant recomputation of key and value tensors for tokens that have already been processed
3. Prohibitive memory and computational costs for practical applications

The key-value cache solves these issues by storing previously computed key-value pairs, dramatically reducing the computational burden during token generation. This foundational technique enables DeepSeek's impressive performance with long contexts of up to 128K tokens.

## 2.1 Understanding Attention Mechanisms

Before diving into the key-value cache, we need to understand the attention mechanism itself. Attention allows a model to focus on relevant parts of the input when producing each element of the output.

### The Attention Formula

The attention mechanism can be expressed mathematically as:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ (queries): What we're looking for
- $K$ (keys): What we match against
- $V$ (values): What we extract when we find a match
- $d_k$: The dimension of the keys (used for scaling)

In transformer models like DeepSeek, each attention layer processes:
- **Queries**: Representations of the token we're currently generating
- **Keys/Values**: Representations of all tokens in the context

### Multi-Head Attention

DeepSeek uses multi-head attention, which allows the model to attend to information from different representation subspaces simultaneously:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

Where each head is computed as:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

This creates multiple "attention heads" that can focus on different aspects of the input sequence.

## 2.2 Autoregressive Generation: The Root of the Inference Bottleneck

DeepSeek models, like other transformer-based LLMs, generate text autoregressively—one token at a time, where each new token depends on all previous tokens. This creates a computational challenge during inference:

1. For the first token, we compute attention using just the prompt
2. For the second token, we compute attention using the prompt plus the first generated token
3. For the third token, we compute attention using all previous tokens
4. And so on...

As the sequence grows, each new token requires more computation than the last. Without optimization, this would create:

- **O(n²) complexity** in sequence length for each new token
- **Redundant calculations** as the same keys and values are recomputed for existing tokens
- **Slow inference speed** for practical applications

The code below demonstrates this autoregressive generation process using a simple GPT-2 model. Pay attention to how we generate one token at a time, repeatedly passing the entire sequence through the model:

In [1]:
#
# SETUP: Imports and Model Loading for the entire notebook
#
import torch
import torch.nn as nn
import time
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Setting up models...")
# Load pre-trained model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
print("Setup complete.")

#
# LISTING 2.1: Visualizing autoregressive generation with GPT-2
#
prompt = "The next day is bright"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids

print(f"Prompt: '{prompt}'", end="")

# Generate 20 tokens
for _ in range(20):
    # Pass the entire sequence to the model
    outputs = model(input_ids)
    logits = outputs.logits

    # Get the logits for the very last token
    next_token_logits = logits[:, -1, :]

    # Get the ID of the most likely next token (greedy decoding)
    next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

    # Append the new token ID to the input sequence
    input_ids = torch.cat([input_ids, next_token_id], dim=-1)

    # Decode and print the new token
    new_token = tokenizer.decode(next_token_id[0])
    print(new_token, end="", flush=True)

print("\n")

Setting up models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setup complete.
Prompt: 'The next day is bright' and sunny, and the sun is shining. The sun is shining, and the moon is shining.



### Listing 2.1: Visualizing Autoregressive Generation

The code below demonstrates the token-by-token generation process in an autoregressive model. We start with a prompt and then generate 20 additional tokens sequentially. Notice how:

1. The entire sequence is passed through the model at each step
2. We extract only the logits for the last token position
3. We select the most likely next token (greedy decoding)
4. We append this token to our sequence and repeat

This naive implementation clearly shows why autoregressive generation becomes increasingly slow as the sequence grows longer. Each token requires a full forward pass through all previous tokens.

In [2]:
#
# LISTING 2.2: Demonstrating the Speedup of KV Caching
#
prompt = "The next day is bright"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

# --- Timing without KV cache ---
print("Generating without KV Cache...")
start_time_without_cache = time.time()
output_without_cache = model.generate(
    input_ids,
    max_new_tokens=100,
    use_cache=False, # Explicitly disable the cache
    attention_mask=attention_mask
)
end_time_without_cache = time.time()
duration_without_cache = end_time_without_cache - start_time_without_cache
print(f"Time without KV Cache: {duration_without_cache:.4f} seconds\n")


# --- Timing with KV cache ---
print("Generating with KV Cache...")
start_time_with_cache = time.time()
output_with_cache = model.generate(
    input_ids,
    max_new_tokens=100,
    use_cache=True, # Explicitly enable the cache
    attention_mask=attention_mask
)
end_time_with_cache = time.time()
duration_with_cache = end_time_with_cache - start_time_with_cache
print(f"Time with KV Cache: {duration_with_cache:.4f} seconds\n")


# --- Calculate and print the speedup ---
speedup = duration_without_cache / duration_with_cache
print(f"KV Cache Speedup: {speedup:.2f}x")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generating without KV Cache...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Time without KV Cache: 57.6215 seconds

Generating with KV Cache...
Time with KV Cache: 6.3934 seconds

KV Cache Speedup: 9.01x


## 2.3 Key-Value Caching: The First Generation Solution

The key-value (KV) cache is the foundational technique for addressing the inference bottleneck in transformer models like DeepSeek. The key insight is simple but powerful:

**Since previous tokens don't change during generation, their key and value projections can be computed once and reused.**

### How KV Caching Works:

1. **Initial Computation**: For the first token, compute Q, K, V as usual
2. **Storage**: Save the K, V tensors in a "cache" 
3. **Subsequent Tokens**: 
   - Only compute Q, K, V for the new token
   - Retrieve previous K, V from the cache
   - Concatenate the new K, V with the cached ones
   - Store the expanded K, V in the cache
4. **Result**: Each new token only needs to compute its own key-value pair rather than recomputing for all tokens

This reduces the computational complexity from O(n²) to O(n) per token, where n is the sequence length.

### Memory-Computation Tradeoff:

The KV cache trades increased memory usage for significantly reduced computation:

- **Memory Usage**: Increases linearly with sequence length
- **Computation**: Dramatically reduced for long sequences

This is particularly important for DeepSeek models with their 128K context window capability.

### Listing 2.2: Benchmarking KV Cache Performance

The code below demonstrates and measures the dramatic speedup achieved through KV caching: